---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 10 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Platform "Kopi Nusantara" (skenario yang sama dari Tugas Mandiri Pertemuan 7) ingin mulai menyimpan **katalog menu** mereka di MongoDB — karena setiap cabang punya menu musiman/spesial dengan atribut yang berbeda-beda (tidak semua menu punya atribut yang sama, misalnya hanya menu kopi yang punya atribut "tingkat_kepahitan", sementara menu makanan punya atribut "pedas_level"). Kalian ditugaskan membangun dan menganalisis katalog ini.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas9_[NIM]_[Nama Lengkap].ipynb`**, koneksikan ke MongoDB, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Membangun Katalog (Create)** *(bobot 20%)*

Buat database baru bernama `kopi_nusantara`, collection bernama `menu`. Sisipkan **minimal 12 dokumen** menu dengan field bervariasi sesuai kategorinya:
- Menu kategori **"Kopi"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Menu kategori **"Makanan"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Boleh menambahkan field lain sesuai kreativitas anda (mis. `stok`, `rekomendasi_barista`).

**B. Kueri Bertingkat (Read)** *(bobot 20%)*

Tampilkan hasil dari **tiga** kueri berikut :
1. Seluruh menu kategori "Kopi" dengan `tingkat_kepahitan` di atas 3.
2. Seluruh menu dengan `harga` di antara 15000 dan 30000.
3. Menu yang **tidak memiliki** field `pedas_level` sama sekali.

**C. Pembaruan Data (Update)** *(bobot 15%)*

Tambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`. Kemudian, ubah `status` menjadi `"Habis"` khusus untuk 2 menu pilihan anda menggunakan `update_one()`.

**D. Integrasi ke Spark** *(bobot 25%)*

Ambil seluruh data dari collection `menu`, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/`null` pada sebagian dokumen, seperti `tingkat_kepahitan` yang hanya dimiliki menu Kopi). Tampilkan:
1. `printSchema()` hasil konversi.
2. Rata-rata `harga` per `kategori` menggunakan `groupBy`.

**E. Kesimpulan** *(bobot 20%)*

Tulis pada markdown cell (**minimal 100 kata**): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas9_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan MongoDB (`sudo systemctl status mongod`) aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Create | Minimal 12 dokumen tersimpan dengan field sesuai ketentuan kategori | 20% |
| B. Read | Ketiga kueri menghasilkan output yang tepat sesuai instruksi | 20% |
| C. Update | Field `status` berhasil ditambahkan & diperbarui sesuai instruksi | 15% |
| D. Integrasi Spark | Konversi ke Spark DataFrame berhasil; analisis `groupBy` benar | 25% |
| E. Kesimpulan | Refleksi menunjukkan pemahaman konseptual yang tepat tentang tantangan skema | 20% |


In [81]:
#Buat koneksi ke MongoDB

from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")

**A. Membangun Katalog (Create)** *(bobot 20%)*

Buat database baru bernama `kopi_nusantara`, collection bernama `menu`. Sisipkan **minimal 12 dokumen** menu dengan field bervariasi sesuai kategorinya:
- Menu kategori **"Kopi"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Menu kategori **"Makanan"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Boleh menambahkan field lain sesuai kreativitas anda (mis. `stok`, `rekomendasi_barista`).

In [82]:
#Buat database dan collection

db = client["kopi_nusantara"]

koleksi_menu = db["menu"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum_bigdata']


In [83]:
#Bersihkan data lama dulu

koleksi_menu.delete_many({})

#Tambahkan dokumen
koleksi_menu.insert_many([
    {"nama_menu": "Americano", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 20},
    {"nama_menu": "Cappuccino", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 15},
    {"nama_menu": "Caffe Latte", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 1, "stok": 25},
    {"nama_menu": "Espresso", "harga": 15000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 10},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 30},
    {"nama_menu": "Mocha", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 12},
    {"nama_menu": "Caramel Macchiato", "harga": 27000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 18},
    {"nama_menu": "Kopi Tubruk", "harga": 12000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 15},
    {"nama_menu": "Mie Goreng", "harga": 15000, "kategori": "Makanan", "pedas_level": 3, "stok": 30},
    {"nama_menu": "Roti Bakar Cokelat", "harga": 12000, "kategori": "Makanan", "pedas_level": 0, "stok": 20},
    {"nama_menu": "Kentang Goreng", "harga": 14000, "kategori": "Makanan", "pedas_level": 1, "stok": 25},
    {"nama_menu": "Nasi Goreng", "harga": 20000, "kategori": "Makanan", "pedas_level": 2, "stok": 18},
    {"nama_menu": "Chicken Katsu", "harga": 25000, "kategori": "Makanan", "pedas_level": 0, "stok": 15},
    {"nama_menu": "Seblak", "harga": 17000, "kategori": "Makanan", "pedas_level": 3, "stok": 22},
    {"nama_menu": "Pisang Goreng", "harga": 13000, "kategori": "Makanan", "pedas_level": 0, "stok": 28}
])

print("Jumlah dokumen di collection 'menu':", koleksi_menu.count_documents({}))

Jumlah dokumen di collection 'menu': 15


**B. Kueri Bertingkat (Read)** *(bobot 20%)*

Tampilkan hasil dari **tiga** kueri berikut :
1. Seluruh menu kategori "Kopi" dengan `tingkat_kepahitan` di atas 3.
2. Seluruh menu dengan `harga` di antara 15000 dan 30000.
3. Menu yang **tidak memiliki** field `pedas_level` sama sekali

In [84]:
#1. 

print("Menu kopi dengan tingkat kepahitan diatas 3:")
for p in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f" - {p['nama_menu']} ({p['tingkat_kepahitan']:,})")

Menu kopi dengan tingkat kepahitan diatas 3:
 - Americano (5)
 - Espresso (5)
 - Kopi Tubruk (4)


In [85]:
#2

print("Menu dengan range harga antara Rp15.000 hingga Rp30.000:")
for p in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"  - {p['nama_menu']} (Rp{p['harga']:,})")

Menu dengan range harga antara Rp15.000 hingga Rp30.000:
  - Americano (Rp18,000)
  - Cappuccino (Rp22,000)
  - Caffe Latte (Rp20,000)
  - Espresso (Rp15,000)
  - Kopi Susu Gula Aren (Rp18,000)
  - Mocha (Rp25,000)
  - Caramel Macchiato (Rp27,000)
  - Mie Goreng (Rp15,000)
  - Nasi Goreng (Rp20,000)
  - Chicken Katsu (Rp25,000)
  - Seblak (Rp17,000)


In [86]:
#3

print("Menu yang tidak memiliki field pedas_level:")
for p in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f" - {p['nama_menu']} (Kategori: {p['kategori']})")

Menu yang tidak memiliki field pedas_level:
 - Americano (Kategori: Kopi)
 - Cappuccino (Kategori: Kopi)
 - Caffe Latte (Kategori: Kopi)
 - Espresso (Kategori: Kopi)
 - Kopi Susu Gula Aren (Kategori: Kopi)
 - Mocha (Kategori: Kopi)
 - Caramel Macchiato (Kategori: Kopi)
 - Kopi Tubruk (Kategori: Kopi)


**C. Pembaruan Data (Update)** *(bobot 15%)*

Tambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`. Kemudian, ubah `status` menjadi `"Habis"` khusus untuk 2 menu pilihan anda menggunakan `update_one()`.


In [87]:
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})

print("Daftar nama menu dan status:")
for p in koleksi_menu.find():
    print(f"- {p['nama_menu']} : {p['status']}")

Daftar nama menu dan status:
- Americano : Tersedia
- Cappuccino : Tersedia
- Caffe Latte : Tersedia
- Espresso : Tersedia
- Kopi Susu Gula Aren : Tersedia
- Mocha : Tersedia
- Caramel Macchiato : Tersedia
- Kopi Tubruk : Tersedia
- Mie Goreng : Tersedia
- Roti Bakar Cokelat : Tersedia
- Kentang Goreng : Tersedia
- Nasi Goreng : Tersedia
- Chicken Katsu : Tersedia
- Seblak : Tersedia
- Pisang Goreng : Tersedia


In [88]:
koleksi_menu.update_one(
    {"nama_menu": "Americano"},
    {"$set": {"status": "Habis"}})

koleksi_menu.update_one(
    {"nama_menu": "Espresso"},
    {"$set": {"status": "Habis"}})

print("Daftar nama menu dan status baru:")
for p in koleksi_menu.find():
    print(f"- {p['nama_menu']} : {p['status']}")

Daftar nama menu dan status baru:
- Americano : Habis
- Cappuccino : Tersedia
- Caffe Latte : Tersedia
- Espresso : Habis
- Kopi Susu Gula Aren : Tersedia
- Mocha : Tersedia
- Caramel Macchiato : Tersedia
- Kopi Tubruk : Tersedia
- Mie Goreng : Tersedia
- Roti Bakar Cokelat : Tersedia
- Kentang Goreng : Tersedia
- Nasi Goreng : Tersedia
- Chicken Katsu : Tersedia
- Seblak : Tersedia
- Pisang Goreng : Tersedia


**D. Integrasi ke Spark** *(bobot 25%)*

Ambil seluruh data dari collection `menu`, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/`null` pada sebagian dokumen, seperti `tingkat_kepahitan` yang hanya dimiliki menu Kopi). Tampilkan:
1. `printSchema()` hasil konversi.
2. Rata-rata `harga` per `kategori` menggunakan `groupBy`.

In [89]:
#1

from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .appName("TugasPertemuan9-MongoDBSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

#Ambil seluruh data dari MongoDB
doc_mongo = list(koleksi_menu.find({}))

#Ubah data MongoDB menjadi Pandas DataFrame
df_pandas = pd.DataFrame(doc_mongo)

#Ubah _id menjadi string dan nilai NaN menjadi None
df_pandas["_id"] = df_pandas["_id"].astype(str)
df_pandas = df_pandas.astype(object).where(pd.notna(df_pandas), None)

#Konversi Pandas DataFrame menjadi Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)

df_spark_menu.printSchema()
df_spark_menu.show()

root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: long (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)

+--------------------+-------------------+-----+--------+-----------------+----+--------+-----------+
|                 _id|          nama_menu|harga|kategori|tingkat_kepahitan|stok|  status|pedas_level|
+--------------------+-------------------+-----+--------+-----------------+----+--------+-----------+
|6ae413a8266cb275c...|          Americano|18000|    Kopi|              5.0|  20|   Habis|       NULL|
|6ae413a8266cb275c...|         Cappuccino|22000|    Kopi|              2.0|  15|Tersedia|       NULL|
|6ae413a8266cb275c...|        Caffe Latte|20000|    Kopi|              1.0|  25|Tersedia|       NULL|
|6ae413a8266cb275c...|           Espresso|15000|    Kopi|          

In [90]:
#2

from pyspark.sql.functions import avg, count

df_spark_menu.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_produk")
).show()

spark.stop()
print("SparkSession ditutup.")

+--------+------------------+-------------+
|kategori|   rata_rata_harga|jumlah_produk|
+--------+------------------+-------------+
|    Kopi|           19625.0|            8|
| Makanan|16571.428571428572|            7|
+--------+------------------+-------------+

SparkSession ditutup.


**E. Kesimpulan** *(bobot 20%)*

Tulis pada markdown cell (**minimal 100 kata**): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?

Tantangan yang saya temui saat mengonversi data MongoDB menjadi Spark DataFrame adalah adanya perbedaan field pada setiap dokumen. MongoDB memiliki struktur data yang fleksibel, sehingga setiap dokumen tidak harus memiliki field yang sama. Pada data menu kafe, misalnya, kategori Kopi memiliki field tingkat_kepahitan, sedangkan kategori Makanan memiliki field pedas_level. Ketika data diubah menjadi Pandas DataFrame, field yang tidak tersedia pada dokumen tertentu akan menghasilkan nilai NaN. Untuk mengatasinya, saya menggunakan Pandas DataFrame sebagai perantara sebelum mengonversi data ke Spark DataFrame. Saya juga mengubah tipe data _id menjadi string menggunakan astype(str) dan mengganti nilai NaN menjadi None menggunakan where(pd.notna(df_pandas), None) agar dapat dikenali sebagai null oleh Spark. Dengan cara tersebut, seluruh field tetap dipertahankan meskipun beberapa dokumen tidak memiliki nilai pada field tertentu. 